# Action0 SNN training foundation

This notebook connects the existing Action0 padded producer output to the three disjoint dataset splits, then renders the pre-training label and valid-frame visualizations. Training is added in a later section.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from snn.action0_dataset import (
    Action0SegmentDataset,
    discover_class_to_idx,
    load_padding_dataset_metadata,
    resolve_dataset_root,
    resolve_segmentation_root,
)
from snn.train_action0 import (
    _validate_sampling_rate,
    _validate_split_padded_lengths,
    _validate_user_splits,
)

In [ ]:
# Keep every notebook-owned setting in this one configuration cell.
PIPELINE_ROOT = Path("outputs/action0_pipeline")
DATASET_VARIANT = "lowpass"
SAMPLE_RATE = 200.0
TRAIN_USERS = [
    "user_0",
    "user_1",
    "user_2",
    "user_3",
    "user_4",
    "user_5",
    "user_6",
    "user_7",
    "user_8",
    "user_9",
    "user_10",
    "user_11",
    "user_12",
    "user_13",
    "user_14",
]
VAL_USERS = ["user_15", "user_16", "user_17"]
TEST_USERS = ["user_18", "user_19", "user_20"]

NEURONS_NETWORK = [24, 24, 24]
SHIFT_SYN = 2
SHIFT_MEM = 1
BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 5e-4
SPIKE_REGULARIZATION = 0.0
RANDOM_SEED = 12345
USE_GPU = False

## Load the producer-published padded dataset

The data foundation uses the producer's `segmentation_padded` packages directly. The shared label mapping is discovered once from the selected variant and passed to every split.

In [ ]:
_validate_user_splits(TRAIN_USERS, VAL_USERS, TEST_USERS)

dataset_root = resolve_dataset_root(PIPELINE_ROOT, DATASET_VARIANT)
segmentation_root = resolve_segmentation_root(dataset_root)

producer_metadata = load_padding_dataset_metadata(segmentation_root)
_validate_sampling_rate(
    cli_sampling_rate=SAMPLE_RATE,
    producer_sampling_rate=producer_metadata.sampling_rate_hz,
)

class_to_idx = discover_class_to_idx(segmentation_root)

train_dataset = Action0SegmentDataset(segmentation_root, TRAIN_USERS, class_to_idx)
val_dataset = Action0SegmentDataset(segmentation_root, VAL_USERS, class_to_idx)
test_dataset = Action0SegmentDataset(segmentation_root, TEST_USERS, class_to_idx)

_validate_split_padded_lengths(
    producer_metadata,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
)

In [ ]:
print(f"Dataset variant: {DATASET_VARIANT}")
print(f"Resolved padded root: {segmentation_root}")
print("Producer metadata:")
print(f"  input_kind: {producer_metadata.input_kind}")
print(f"  feature_schema: {producer_metadata.feature_schema}")
print(f"  channel_count: {producer_metadata.channel_count}")
print(f"  target_length: {producer_metadata.target_length}")
print(f"  sampling_rate_hz: {producer_metadata.sampling_rate_hz:g}")
print(f"  padding_side: {producer_metadata.padding_side}")
print("Model-facing feature channels: 0:15")
print(f"Global class mapping ({len(class_to_idx)} classes): {class_to_idx}")

for split_name, dataset in (
    ("train", train_dataset),
    ("validation", val_dataset),
    ("test", test_dataset),
):
    print(
        f"{split_name}: len={len(dataset)}, "
        f"class_distribution={dataset.class_distribution}, "
        f"padded_length={dataset.padded_length}"
    )

## Pre-training label distributions

Percentages use each dataset's segment count and class distribution. Labels follow the shared global class-map order, so a class absent from a split is shown as zero.

In [ ]:
labels = list(class_to_idx)
split_datasets = {
    "Train": train_dataset,
    "Val": val_dataset,
    "Test": test_dataset,
}

percentage_by_split = {
    split_name: [
        100.0 * dataset.class_distribution.get(label, 0) / len(dataset)
        for label in labels
    ]
    for split_name, dataset in split_datasets.items()
}

print("Label | Train % | Val % | Test %")
for label_index, label in enumerate(labels):
    print(
        f"{label} | {percentage_by_split['Train'][label_index]!r} | "
        f"{percentage_by_split['Val'][label_index]!r} | "
        f"{percentage_by_split['Test'][label_index]!r}"
    )

for split_name, percentages in percentage_by_split.items():
    figure, axis = plt.subplots(figsize=(12, 6))
    axis.bar(labels, percentages)
    axis.set_title(f"{split_name} label distribution")
    axis.set_xlabel("Original label")
    axis.set_ylabel("Segments (%)")
    axis.tick_params(axis="x", rotation=90)
    axis.set_ylim(0, max(100.0, max(percentages, default=0.0)))
    figure.tight_layout()
    plt.show()

## Valid-frame 15-channel segment view

This deterministic view uses `train_dataset[0]`. Its horizontal axis is relative nominal sample-index time computed from the declared producer rate; acquisition timestamps are not loaded or inferred, and right-padding is excluded by the producer mask.

In [ ]:
idx_to_class = {index: label for label, index in class_to_idx.items()}
segment_features, segment_label_index, valid_mask = train_dataset[0]
segment_label = idx_to_class[int(segment_label_index.item())]

valid_indices = np.flatnonzero(valid_mask.numpy())
relative_nominal_time_s = valid_indices / producer_metadata.sampling_rate_hz
segment_features_np = segment_features.numpy()

segment_figure, segment_axes = plt.subplots(
    nrows=15,
    ncols=1,
    sharex=True,
    figsize=(12, 24),
    squeeze=False,
)
for channel_index in range(15):
    axis = segment_axes[channel_index, 0]
    axis.plot(
        relative_nominal_time_s,
        segment_features_np[valid_indices, channel_index],
    )
    axis.set_ylabel(f"Ch {channel_index}\nspike amplitude")
    axis.grid(True, alpha=0.3)

segment_axes[-1, 0].set_xlabel(
    "Relative nominal sample-index time = valid_indices / declared producer sampling_rate_hz [s]"
)
segment_figure.suptitle(
    f"Train segment 0 — original label: {segment_label} (valid frames only)"
)
segment_figure.tight_layout(rect=(0, 0, 1, 0.99))
plt.show()

## Existing SynNet training loop

The training section reuses the Action0 model, masked loss, dataloaders, and epoch engine. Metrics and the strict-greater best validation state remain in memory for the result section.

In [ ]:
from copy import deepcopy
from types import SimpleNamespace

import torch

from snn.action0_engine import run_epoch
from snn.action0_losses import MaskedCrossEntropySpkReg
from snn.train_action0 import (
    _create_dataloaders,
    _resolve_device,
    _set_random_seed,
    _validate_args,
)
from snn.utils_architectures import SynNet

if len(NEURONS_NETWORK) != 3:
    raise ValueError("NEURONS_NETWORK must contain exactly three SynNet layers")

training_args = SimpleNamespace(
    network_type="SynNet",
    sample_freq=SAMPLE_RATE,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    spike_regularization=SPIKE_REGULARIZATION,
    num_workers=0,
    max_train_batches=None,
    shift_syn=SHIFT_SYN,
    shift_mem=SHIFT_MEM,
    neurons_network=NEURONS_NETWORK,
    train_users=TRAIN_USERS,
    val_users=VAL_USERS,
    test_users=TEST_USERS,
)
_validate_args(training_args)

_set_random_seed(RANDOM_SEED)
device = _resolve_device(USE_GPU)
train_loader, val_loader, test_loader = _create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
    random_seed=RANDOM_SEED,
    device=device,
)

preview_inputs, preview_labels, preview_valid_mask = next(iter(train_loader))
print(
    "Preview batch: inputs={}, labels={}, valid_mask={}".format(
        tuple(preview_inputs.shape),
        tuple(preview_labels.shape),
        tuple(preview_valid_mask.shape),
    )
)

model = SynNet(
    inputSize=15,
    outputSize=len(class_to_idx),
    hiddenSizes=NEURONS_NETWORK,
    sampleFreq=SAMPLE_RATE,
    shiftSyn=SHIFT_SYN,
    shiftMem=SHIFT_MEM,
).to(device)
criterion = MaskedCrossEntropySpkReg(
    spike_regularization=SPIKE_REGULARIZATION
)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

_HISTORY_METRICS = (
    "loss",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "mean_output_spikes",
    "mean_total_spikes",
    "zero_output_spike_fraction",
)
history = {
    "epochs": [],
    "train": {metric_name: [] for metric_name in _HISTORY_METRICS},
    "val": {metric_name: [] for metric_name in _HISTORY_METRICS},
}
best_state = None
best_epoch = -1
best_val_balanced_accuracy = float("-inf")

for epoch in range(NUM_EPOCHS):
    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        split="train",
        device=device,
        expected_num_classes=len(class_to_idx),
    )
    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        split="val",
        device=device,
        expected_num_classes=len(class_to_idx),
    )

    history["epochs"].append(epoch)
    for split_name, split_metrics in (("train", train_metrics), ("val", val_metrics)):
        for metric_name in _HISTORY_METRICS:
            history[split_name][metric_name].append(float(split_metrics[metric_name]))

    validation_balanced_accuracy = float(val_metrics["balanced_accuracy"])
    if validation_balanced_accuracy > best_val_balanced_accuracy:
        best_val_balanced_accuracy = validation_balanced_accuracy
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS}: "
        f"train loss={train_metrics['loss']:.6f}, "
        f"val balanced_accuracy={validation_balanced_accuracy:.6f}"
    )

if best_state is None:
    raise RuntimeError("training completed without a validation result")
for split_name in ("train", "val"):
    for metric_name in _HISTORY_METRICS:
        assert len(history[split_name][metric_name]) == NUM_EPOCHS
assert len(history["epochs"]) == NUM_EPOCHS
print(
    f"Best validation balanced accuracy: {best_val_balanced_accuracy:.6f} "
    f"at zero-based epoch {best_epoch}"
)